# Extracción de datos (Notebook de demostración)
#### Extraemos los paraderos de https://github.com/JoseDTPM/geojson-Transantiago, que posee archivos .json actualizados a 2024, de los paraderos de Santiago y sus coordenadas.

In [2]:
# Creamos un dataframe que tenga todos los ID de paraderos de Santiago usando
# https://raw.githubusercontent.com/JoseDTPM/geojson-Transantiago/refs/heads/main/Paraderos-Santiago-Chile.geojsonl.json

from os import listdir
listdir("data")
import pandas as pd

file = "data/Paraderos-Santiago-Chile.geojsonl.json"
df = pd.read_json(file, lines=True)
properties_df = pd.json_normalize(df["properties"])
geometry_df = pd.json_normalize(df["geometry"])
coordinates_df = geometry_df[["coordinates"]]
print(coordinates_df)
# Seleccionamos la columna que corresponde al ID de los paraderos.
stops_df = properties_df[["SIMT"]]
stops_df["coordinates"] = coordinates_df
stops_df_clean = pd.DataFrame()
stops_df_clean["SIMT"] = stops_df["SIMT"].unique()

                    coordinates
0      [-70.823088, -33.572525]
1        [-70.8189, -33.571649]
2       [-70.818197, -33.57202]
3       [-70.817967, -33.57187]
4      [-70.816035, -33.574243]
...                         ...
11793  [-70.506901, -33.332828]
11794  [-70.499667, -33.356539]
11795   [-70.499282, -33.34372]
11796  [-70.495669, -33.342486]
11797  [-70.496589, -33.340103]

[11798 rows x 1 columns]


/tmp/ipykernel_8747/3629709076.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stops_df["coordinates"] = coordinates_df


#### Creamos un dataframe con el código de los paraderos y sus coordenadas.

In [3]:
exploded_df = stops_df.copy()
exploded_df[['Latitud','Longitud']] = pd.DataFrame(exploded_df["coordinates"].tolist(), index=exploded_df.index)
exploded_df = exploded_df[['SIMT', 'Latitud', 'Longitud']]
exploded_df

,SIMT,Latitud,Longitud
0,PI1923,-70.823088,-33.572525
1,PI1935,-70.818900,-33.571649
2,PI1926,-70.818197,-33.572020
3,PI1925,-70.817967,-33.571870
4,PI1927,-70.816035,-33.574243
...,...,...,...
11793,PC861,-70.506901,-33.332828
11794,PC596,-70.499667,-33.356539
11795,PC918,-70.499282,-33.343720
11796,PC919,-70.495669,-33.342486


#### Limpiamos paraderos duplicados (si es que existen), y posteriormente, tomamos un ejemplo de 1000 paraderos aleatorios que serán analizados. Este es un límite de API.

In [4]:
stops_df_1000 = exploded_df.sample(n = 1000).reset_index()
print(f"cant. normal: {len(stops_df)}\ncant. limpio: {len(stops_df_clean)}\ncant. a analizar: {len(stops_df_1000)}")
stops_df_1000

cant. normal: 11798
cant. limpio: 11798
cant. a analizar: 1000


,index,SIMT,Latitud,Longitud
0,143,PG760,-70.713975,-33.597329
1,8365,PE877,-70.582898,-33.550203
2,4216,PH475,-70.699484,-33.526467
3,1543,PF821,-70.583326,-33.603785
4,338,PG745,-70.707052,-33.589723
...,...,...,...,...
995,11217,PC125,-70.553337,-33.420680
996,3832,PI539,-70.724512,-33.472924
997,5300,PJ758,-70.710891,-33.439529
998,7752,PE433,-70.638469,-33.527743


#### Luego, procedemos a extraer los datos para almacenarlos en un archivo CSV.

In [6]:
# Creamos un dataframe que tenga todos los ID de paraderos de Santiago usando
# https://raw.githubusercontent.com/JoseDTPM/geojson-Transantiago/refs/heads/main/Paraderos-Santiago-Chile.geojsonl.json

import pandas as pd
import requests
import time
from datetime import datetime
from os import listdir

def bus_routes(stop):
    all_routes_data = []
    total_stops = len(stop)
    i = 0
    # Iteramos por código de paradero
    for code in stop["SIMT"]:
        #LIMITE DE EJEMPLO PARA ESTE NOTEBOOK
        if i > 5:
            break
        i += 1
        print(f"Paradero actual: {code}")
        url = f"https://api.xor.cl/red/bus-stop/{code}"
        try:
            response = requests.get(url)
        # Cuando la API rechace las solicitudes, cortamos el for y creamos el CSV de todas maneras.
        except:
            break
        data = response.json()
        # Obtenemos los datos relevantes
        for service in data.get("services", []):
                for bus in service.get("buses", []):
                    current_datetime = datetime.now()
                    all_routes_data.append({
                        "bus_stop_code": code,
                        "route_id": service.get("id"),
                        "bus_id": bus.get("id"),
                        "meters_distance": bus.get("meters_distance"),
                        "min_arrival_time": bus.get("min_arrival_time"),
                        "max_arrival_time": bus.get("max_arrival_time"),
                        "date": current_datetime
            })
    # Creamos un dataframe para cada recorrido
    bus_routes = pd.DataFrame(all_routes_data)
    # Juntamos el dataframe de los buses con el de los paraderos
    final_df = pd.merge(
    bus_routes,
    stops_df,
    left_on="bus_stop_code",
    right_on="SIMT",
    how="left"
    )

    final_df = final_df.drop(columns=["SIMT"])
    return final_df

file = "data/Paraderos-Santiago-Chile.geojsonl.json"
df = pd.read_json(file, lines=True)
properties_df = pd.json_normalize(df["properties"])

# Seleccionamos la columna que corresponde al ID de los paraderos y sus coordenadas.
stops_df = properties_df[["SIMT"]]
stops_df_clean = pd.DataFrame()
stops_df_clean["SIMT"] = stops_df["SIMT"].unique()
stops_df_1000 = stops_df.sample(n = 1000).reset_index()
# Creamos un string con la fecha y hora de hoy
timeanddate = datetime.now()
timeanddate = timeanddate.strftime("%d-%m-%Y-%H-%M")


final_bus_df = bus_routes(stops_df_1000)
final_bus_df
# Exportamos como CSV
#final_bus_df.to_csv(f"buses_outputs/datos_{timeanddate}.csv", index=False)
#print("CSV creado.")

Paradero actual: PD351
Paradero actual: PJ659
Paradero actual: PI372
Paradero actual: PD228
Paradero actual: PA592
Paradero actual: PF743


,bus_stop_code,route_id,bus_id,meters_distance,min_arrival_time,max_arrival_time,date
0,PD351,D02,SVDC-61,2103,0,6,2025-10-21 23:19:02.745309
1,PJ659,J02,VHXJ-22,3075,0,8,2025-10-21 23:19:04.795290
2,PJ659,J02,VHXJ-10,10066,21,29,2025-10-21 23:19:04.795304
3,PD228,D15,SPZK-84,3869,17,25,2025-10-21 23:19:09.198957
4,PD228,D15,STRJ-56,10200,33,43,2025-10-21 23:19:09.198971
5,PD228,216,LZPG-10,285,0,3,2025-10-21 23:19:09.198974
6,PD228,216,SFST-37,5236,10,16,2025-10-21 23:19:09.198976
7,PD228,712,LCPW-29,2314,0,6,2025-10-21 23:19:09.198977
8,PD228,712,SHCW-65,7794,14,20,2025-10-21 23:19:09.198979
9,PA592,345,CJRW-11,735,0,3,2025-10-21 23:19:11.354035


#### Ahora, extraemos la información de la red de metro, proveniente de https://api.xor.cl/red/metro-network.

In [6]:
def get_metro_data():
    url = "https://api.xor.cl/red/metro-network"
    all_stations_data = []

    response = requests.get(url)
    lines_data = response.json()

    lines_list = lines_data.get("lines", [])
    # Iteramos por linea
    for line in lines_list:
        line_id = line.get("id")
        
        stations = line.get("stations", [])
        # Iteramos por estación
        for station in stations:
            # Determinamos la fecha actual
            current_datetime = datetime.now()
            all_stations_data.append({
                "line_id": line_id,
                "station_id": station.get("id"),
                "name": station.get("name"),
                "status_code": station.get("status"),
                "status_description": station.get("description"),
                "date": current_datetime
            })
    # Retornamos el dataframe del estado de las estaciones
    return pd.DataFrame(all_stations_data)

timeanddate = datetime.now()
timeanddate = timeanddate.strftime("%d-%m-%Y-%H-%M")
    
metro_df = get_metro_data()
metro_df
# Exportamos como CSV
#metro_df.to_csv(f"metro_outputs/metro_{timeanddate}.csv", index=False)

,line_id,station_id,name,status_code,status_description,date
0,L1,san-pablo,San Pablo,0,Estación Operativa,2025-10-21 19:02:54.312674
1,L1,neptuno,Neptuno,0,Estación Operativa,2025-10-21 19:02:54.312684
2,L1,pajaritos,Pajaritos,0,Estación Operativa,2025-10-21 19:02:54.312686
3,L1,las-rejas,Las Rejas,0,Estación Operativa,2025-10-21 19:02:54.312687
4,L1,ecuador,Ecuador,0,Estación Operativa,2025-10-21 19:02:54.312687
...,...,...,...,...,...,...
138,L6,nuble,Ñuble,0,Estación Operativa,2025-10-21 19:02:54.312764
139,L6,estadio-nacional,Estadio Nacional,0,Estación Operativa,2025-10-21 19:02:54.312764
140,L6,nunoa,Ñuñoa,0,Estación Operativa,2025-10-21 19:02:54.312765
141,L6,ines-de-suarez,Inés de Suárez,0,Estación Operativa,2025-10-21 19:02:54.312765
